STEP 1: Data Aggregation

scraping treatment data from EIA API

In [ ]:
import numpy as np
import requests
import pandas as pd
from bs4 import BeautifulSoup
import io

In [ ]:
# extract list of state codes from EIA website
url = "https://www.eia.gov/dnav/pet/pet_pri_dfp1_k_m.htm"
page = requests.get(url)
soup = BeautifulSoup(page.text, "html.parser")

exclude_codes = ["F009960", "F001236", "F003075", "F005071", "F005061", "F005074"]

codes = []
for link in soup.find_all("a"):
    href = link.get("href", "")
    text = link.text.strip()
    if "s=F" in href and "&f=M" in href:
        code = href.split("s=")[-1].split("&")[0]
        num_part = int(code[1:7])
        if num_part % 1000 == 0:
          continue
        if code[:7] in exclude_codes:
          continue
        codes.append(code)
print(codes)

In [ ]:
all_data = []
API_URL = "https://api.eia.gov/v2/petroleum/pri/dfp1/data/"

# call the EIA API
for code in codes:
    params = {
        "api_key": "vUc8dcxSykilX0CpePaERE6F0NRipDBTqiB1Jl51",
        "frequency": "monthly",
        "data[0]": "value",
        "facets[series][]": code,
        "sort[0][column]": "period",
        "sort[0][direction]": "desc",
        "offset": 0,
        "length": 5000
    }
    try:
      response = requests.get(API_URL, params=params)
      response.raise_for_status()
      data = response.json()

      # Extract records
      for r in data['response']['data']:
          all_data.append({
              "State_Code": code,
              "Period": r['period'],
              "Price": r['value']
          })
    except Exception as e:
      print(f"Error fetching data for code {code}: {e}")

# Convert to DataFrame and process as before
df = pd.DataFrame(all_data)
df['Period'] = pd.to_datetime(df['Period'])
df['Year'] = df['Period'].dt.year
df['Month'] = df['Period'].dt.month
df = df[(df['Year'] >= 1978) & (df['Year'] <= 2024)]
df.sort_values(by=['State_Code', 'Year', 'Month'], inplace=True)

df.head(10)

In [ ]:
# data cleaning: remove rows where price is withdrawn
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df = df.dropna(subset=['Price'])
df = df.reset_index(drop=True)

In [ ]:
# add state column
state_mapping = {
    'F001242__3': 'PA',
    'F001354__3': 'WV',
    'F002017__3': 'IL',
    'F002018__3': 'IN',
    'F002020__3': 'KS',
    'F002021__3': 'KY',
    'F002026__3': 'MI',
    'F002031__3': 'NE',
    'F002038__3': 'ND',
    'F002039__3': 'OH',
    'F002040__3': 'OK',
    'F002046__3': 'SD',
    'F003001__3': 'AL',
    'F003005__3': 'AR',
    'F003022__3': 'LA',
    'F003028__3': 'MS',
    'F003035__3': 'NM',
    'F003048__3': 'TX',
    'F004008__3': 'CO',
    'F004030__3': 'MT',
    'F004049__3': 'UT',
    'F004056__3': 'WY',
    'F005006__3': 'CA'
}

# get state mappings
df['state'] = df['State_Code'].map(state_mapping)
print(df.head(10))
print(df.shape)

Join outcome variable data from external data source: https://www.eia.gov/electricity/data/state/

In [ ]:
cons_csv = pd.read_csv('data/consumption_data.csv')
cons_csv.head(20)

In [ ]:
# filter for only petroleum derived electricity consumption
cons_csv = cons_csv[(cons_csv['ENERGY SOURCE              (UNITS)'].str.contains('Petroleum')) & (cons_csv['TYPE OF PRODUCER']=='Total Electric Power Industry')]
print(cons_csv.head())
print(cons_csv.shape)

In [ ]:
# join the two tables
cons_csv['YEAR'] = cons_csv['YEAR'].astype('int32')
cons_csv['MONTH'] = cons_csv['MONTH'].astype('int32')
df = df.merge(cons_csv, left_on=['Year', 'state', 'Month'], right_on=['YEAR', 'STATE', 'MONTH'], how='inner')
df = df.drop(columns=['YEAR', 'STATE', 'MONTH'])
print(df.head())
print(df.shape)

Join confounder yield curve data from external source: https://fred.stlouisfed.org/series/T10Y2Y

In [ ]:
yield_curve = pd.read_csv('data/treasury_data.csv')
yield_curve.head()

In [ ]:
# join with existing df
yield_curve.rename(columns={'T10Y2YM':'Treasury Yield Spread (10Y-2Y)'}, inplace=True)
yield_curve['Year'] = yield_curve['observation_date'].str.split('-').str[0]
yield_curve.drop(columns=['observation_date'], inplace=True)
yield_curve['Year'] = yield_curve['Year'].astype('int32')

df = df.merge(yield_curve, on='Year', how='inner')
print(df.head())
print(df.shape)

Join confounder inflation rate data from external source: https://fred.stlouisfed.org/series/FPCPITOTLZGUSA

In [ ]:
inflation = pd.read_csv('data/inflation_data.csv')
inflation.head()

In [ ]:
# join with existing df
inflation.rename(columns={'FPCPITOTLZGUSA':'Inflation Rate (%)'}, inplace=True)
inflation['Year'] = inflation['observation_date'].str.split('-').str[0]
inflation.drop(columns=['observation_date'], inplace=True)
inflation['Year'] = inflation['Year'].astype('int32')

df = df.merge(inflation, on='Year', how='inner')
print(df.head())
print(df.shape)

Join confounder industrial production index data from external source: https://fred.stlouisfed.org/series/INDPRO

In [ ]:
ind_idx = pd.read_csv('data/industrial_production_data.csv')
ind_idx.head()

In [ ]:
# join with existing df
ind_idx.rename(columns={'INDPRO':'Industrial Production Index'}, inplace=True)
ind_idx['Year'] = ind_idx['observation_date'].str.split('-').str[0]
ind_idx['Month'] = ind_idx['observation_date'].str.split('-').str[1]
ind_idx.drop(columns=['observation_date'], inplace=True)
ind_idx['Year'] = ind_idx['Year'].astype('int32')
ind_idx['Month'] = ind_idx['Month'].astype('int32')

df = df.merge(ind_idx, on=['Year', 'Month'], how='inner')
print(df.head())
print(df.shape)

Join confounder weather data by calling NOAA API

In [ ]:
# integrate temperature on monthly state-level basis
import time

# call NOAA endpoint to extract weather data
noaa_url = "https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/statewide/time-series/{state}/tavg/1/0/2000-2025.json"
all_data = []

for state_id in range(1, 51):
    url = noaa_url.format(state=state_id)
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        j = r.json()
        for date, value in j["data"].items():
            year = int(date[:4])
            month = int(date[4:])
            all_data.append({
                "state_id": state_id,
                "year": year,
                "month": month,
                "temperature": value
            })

    except Exception as e:
        print(f"State {state_id} failed: {e}")

    time.sleep(0.4)

tmp = pd.DataFrame(all_data)
print(tmp.head())
print(tmp.shape)

In [ ]:
def extract_temp(val):
  return val['value']

tmp['temperature'] = tmp['temperature'].apply(extract_temp)

state_id_to_code = {
    1: "AL", 2: "AZ", 3: "AR", 4: "CA", 5: "CO",
    6: "CT", 7: "DE", 8: "FL", 9: "GA", 10: "HI",
    11: "ID", 12: "IL", 13: "IN", 14: "IA", 15: "KS",
    16: "KY", 17: "LA", 18: "ME", 19: "MD", 20: "MA",
    21: "MI", 22: "MN", 23: "MS", 24: "MO", 25: "MT",
    26: "NE", 27: "NV", 28: "NH", 29: "NJ", 30: "NM",
    31: "NY", 32: "NC", 33: "ND", 34: "OH", 35: "OK",
    36: "OR", 37: "PA", 38: "RI", 39: "SC", 40: "SD",
    41: "TN", 42: "TX", 43: "UT", 44: "VT", 45: "VA",
    46: "WA", 47: "WV", 48: "WI", 49: "WY", 50: "AK"
}

# map code to existing state abbrev
tmp['State'] = tmp['state_id'].map(state_id_to_code)
tmp.drop(columns=['state_id'], inplace=True)
tmp.head()

In [ ]:
# merge with existing df
tmp.rename(columns={'temperature':'temperature (F)','year':'Year','month':'Month','State':'state'}, inplace=True)
tmp['Year'] = tmp['Year'].astype('int32')
tmp['Month'] = tmp['Month'].astype('int32')
df = df.merge(tmp, on=['state', 'Year', 'Month'], how='inner')
print(df.head())
print(df.shape)

Join confounder electricity price data from external source: https://www.eia.gov/state/seds/seds-data-complete.php

In [ ]:
prices = pd.read_csv('data/electricity_price_data.csv')
prices.head()

In [ ]:
# remove empty columns
remove_colums = []
for col in prices.columns:
  if 'Unnamed' in col:
    remove_colums.append(col)

prices.drop(columns=remove_colums, inplace=True)

prices = prices.melt(
    id_vars=["State"],        # Columns to keep
    var_name="Year",          # Name for the new "variable" column
    value_name="Price"        # Name for the new "value" column
)

# Convert Year and Price to numeric
prices["Year"] = prices["Year"].astype(int)
prices["Price"] = pd.to_numeric(prices["Price"], errors='coerce')

print(prices.head())

In [ ]:
# join with existing df
prices.rename(columns={'Price':'Electricity Price ($ per million BTU)','State':'state'}, inplace=True)
prices['Year'] = prices['Year'].astype('int32')
df = df.merge(prices, on=['state', 'Year'], how='inner')
print(df.head())
print(df.shape)

In [ ]:
opec = pd.read_csv('data/OPEC_data.csv')
opec.head()

In [ ]:
# extract years as distinct rows
opec = pd.melt(opec, var_name='Year', value_name='OPEC supply (1000 b/d)')
opec.head()

In [ ]:
opec['Year'] = opec['Year'].astype(int)
df = df.merge(opec, on='Year')
df.head()

In [ ]:
# save aggregated data
df.to_csv('data/data102_oil_data_final.csv')

STEP 2: generate visualizations for EDA section

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# data cleaning: changing colum dtypes
df['CONSUMPTION'] = pd.to_numeric(
    df['CONSUMPTION'].astype(str).str.replace(',', ''),
    errors='coerce'
).astype(pd.Int64Dtype())
df['OPEC supply (1000 b/d)'] = pd.to_numeric(
    df['OPEC supply (1000 b/d)'].astype(str).str.replace(',', ''),
    errors='coerce'
).astype(pd.Int64Dtype())

In [ ]:
# visualize treatment vs outcome
plt.figure(figsize=(10, 6))
plt.scatter(df['Price'], df['CONSUMPTION'], alpha=0.1)
plt.xlabel('Price ($/barrel)')
plt.ylabel('Consumption (barrels)')
plt.title('Oil Price vs. Petroleum-derived Electricity Consumption')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# visualize treatment - outcome correlations state by state
state_vals = df['state'].unique()
corr_vals = []
for curr_val in state_vals:
  curr_df = df[df['state'] == curr_val]
  corr_vals.append(curr_df['Price'].corr(curr_df['CONSUMPTION']))

plt.hist(corr_vals, bins=10)
plt.xlabel('Correlation Value')
plt.ylabel('Frequency')
plt.title('Histogram of Treatement-Target Correlation Values by State')
plt.show()

In [ ]:
# visualize confounder - outcome relationships
target = 'CONSUMPTION'
exclude_features = ['Period', 'Year', 'Month', 'CONSUMPTION', 'state', 'State_Code']
features = [col for col in df.columns if col not in exclude_features]
features = features[-6:]

nrows, ncols = 3, 2
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 12))
axes = axes.flatten()

num_plots = len(features)

for i in range(num_plots):
    feature = features[i]
    ax = axes[i]

    sns.scatterplot(x=df[feature], y=df[target], alpha=0.4, ax=ax)

    ax.set_title(f'{feature} vs. Consumption', fontsize=14)
    ax.set_xlabel(feature, fontsize=12)
    ax.set_ylabel('Consumption (barrels)', fontsize=12)

for j in range(num_plots, nrows * ncols):
    if j < len(axes):
        fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

STEP 3: perform modeling

In [ ]:
# one hot encode state column
df = pd.get_dummies(df, columns=['state'], dtype=int)
df.head()

In [ ]:
for col in df.columns:
  print(f"column: {col}, type: {type(df[col].iloc[0])}")

In [ ]:
# apply seasonality feature
model_df = df.copy()
model_df['time_idx'] = (model_df['Year'] - 2000) * 12 + (model_df['Month'] - 1)
model_df['month_sin'] = np.sin(2 * np.pi * model_df['Month'] / 12)
model_df['month_cos'] = np.cos(2 * np.pi * model_df['Month'] / 12)
model_df.rename(columns={'ENERGY SOURCE              (UNITS)': 'ENERGY SOURCE (UNITS)'}, inplace=True)

# Set to ordered format
model_df['Period'] = pd.to_datetime(model_df['Period'])

model_df.drop(columns=['State_Code', 'TYPE OF PRODUCER', 'ENERGY SOURCE (UNITS)'], inplace=True)
print(model_df.info())

In [ ]:
from sklearn.model_selection import train_test_split
import statsmodels.api as sm

In [ ]:
treatment = 'Price'
outcome = 'CONSUMPTION'
ate_df = model_df.copy()
treatment_med = ate_df['Price'].median()
T = np.where(ate_df['Price'] >= treatment_med, 1, 0)
ate_df.drop(columns=['Period', 'Year', 'Month'], inplace=True)
features = [col for col in ate_df.columns if col not in ['CONSUMPTION', 'Price']]

# derive propensity scores
X_confounders = ate_df[features]
X_confounders_sm = sm.add_constant(X_confounders)
logit_model = sm.Logit(T, X_confounders_sm)
logit_results = logit_model.fit(disp=0)
propensity_scores = logit_results.predict(X_confounders_sm)
ate_df['propensity_score'] = propensity_scores
ate_df['T'] = T

# perform trimming on extreme values
lower, upper = 0.025, 0.975
mask = (
    (ate_df['propensity_score'] > lower) &
    (ate_df['propensity_score'] < upper)
)

ate_trim = ate_df.loc[mask].copy()
Y = ate_trim[outcome].values
T_trim = ate_trim['T'].values
e = ate_trim['propensity_score'].values
N = len(ate_trim)
# derive ATE
ate_ht = (1 / N) * np.sum(
    (T_trim * Y / e) - ((1 - T_trim) * Y / (1 - e))
)

print(f"Horvitz–Thompson IPW ATE (HT): {ate_ht:.4f}")